In [1]:
import sys
import json
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent.parent.parent))

print("✓ Setup complete")

✓ Setup complete


## Load Data

In [2]:
# Check if data files exist
from pathlib import Path

data_dir = Path("data")
labels_file = data_dir / "labels.json"
summary_file = data_dir / "summarization_results.json"

if not labels_file.exists():
    print(f"⚠ Warning: {labels_file} not found")
    print("  Please ensure data files are in the correct location")
else:
    print(f"✓ Found {labels_file}")

if not summary_file.exists():
    print(f"⚠ Warning: {summary_file} not found")
    print("  Please ensure data files are in the correct location")
else:
    print(f"✓ Found {summary_file}")

if labels_file.exists() and summary_file.exists():
    print("\n✓ All data files present and ready")

✓ Found data/labels.json
✓ Found data/summarization_results.json

✓ All data files present and ready


In [3]:
try:
    # Load labels (entity extraction results)
    with open('data/labels.json', 'r') as f:
        labels_data = json.load(f)

    # Load summarization results
    with open('data/summarization_results.json', 'r') as f:
        summary_data = json.load(f)

    # Extract entities from labels
    entities = {}
    for page in labels_data:
        entities.update(page["entity_value"])

    # Extract summary
    summary = summary_data["summary"]

    print(f"✓ Loaded {len(labels_data)} pages with entity data")
    print(f"✓ Total entities: {len(entities)}")
    print(f"✓ Summary length: {len(summary)} characters")
    
    print("\nEntities:")
    for k, v in entities.items():
        print(f"  {k}: {v}")

    print(f"\nSummary:\n{summary}")
    
except FileNotFoundError as e:
    print(f"❌ Error: Data file not found - {e}")
    print("Please ensure data files exist in the 'data' directory")
except json.JSONDecodeError as e:
    print(f"❌ Error: Invalid JSON format - {e}")
except KeyError as e:
    print(f"❌ Error: Missing expected key in data - {e}")
    print("Please verify data file structure")

✓ Loaded 4 pages with entity data
✓ Total entities: 5
✓ Summary length: 192 characters

Entities:
  Applicant Name: Ms LOK WING CHING
  Job Title of Applicant: DIRECTOR
  Business Registration Number of Employer: 21893829
  Height of Applicant: 174 cm
  Weight of Applicant: 77 kg

Summary:
The applicant, Ms. Lok Wing Ching, is currently employed as a Director. Her employer’s business registration number is 21893829. Ms. Lok’s height is recorded as 174 cm and her weight is 77 kg.


## Initialize Evaluators

In [4]:
from evaluation.document_summarization.evaluators import (
    EntityCoverageEvaluator,
    GroundednessEvaluator,
    SemanticFidelityEvaluator
)

# Create evaluator instances
ecs_evaluator = EntityCoverageEvaluator()
gs_evaluator = GroundednessEvaluator()
sef_evaluator = SemanticFidelityEvaluator()

print("✓ All evaluators initialized")

✓ All evaluators initialized


## 1. Entity Coverage Score (ECS)

Measures how much of the known entities made it into the summary.

In [5]:
ecs_result = ecs_evaluator.evaluate(summary=summary, entities=entities)

print(f"Entity Coverage Score: {ecs_result.score:.3f}")
print(f"Feedback: {ecs_result.feedback}")
print(f"\nCovered Entities ({len(ecs_result.metadata['covered_entities'])}): {ecs_result.metadata['covered_entities']}")
print(f"Missing Entities ({len(ecs_result.metadata['missing_entities'])}): {ecs_result.metadata['missing_entities']}")
print(f"\nWeights (Top 5):")
for entity, weight in sorted(ecs_result.metadata['weights'].items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {entity}: {weight:.3f}")

Entity Coverage Score: 1.000
Feedback: Covered 5/5 entities (100.0%)

Covered Entities (5): ['Applicant Name', 'Job Title of Applicant', 'Business Registration Number of Employer', 'Height of Applicant', 'Weight of Applicant']
Missing Entities (0): []

Weights (Top 5):
  Applicant Name: 2.361
  Job Title of Applicant: 1.792
  Business Registration Number of Employer: 1.792
  Height of Applicant: 1.725
  Weight of Applicant: 1.648


## 2. Groundedness Score (GS)

Checks if facts in the summary are grounded in source entities using:
- **Numeric entities**: Relative deviation penalty
- **Textual entities**: Token containment (support-based, no fuzzy matching)

In [6]:
gs_result = gs_evaluator.evaluate(summary=summary, entities=entities)

print(f"Groundedness Score: {gs_result.score:.3f}")
print(f"Feedback: {gs_result.feedback}")
print(f"\nEntity-Level Scores:")
for entity, score in gs_result.metadata['entity_scores'].items():
    print(f"  {entity}: {score:.3f}")
print(f"\nNumeric Entities: {gs_result.metadata['numeric_entities']}")
print(f"Textual Entities: {gs_result.metadata['textual_entities']}")
print(f"Well Grounded: {gs_result.metadata['well_grounded_entities']}")
print(f"Poorly Grounded: {gs_result.metadata['poorly_grounded_entities']}")

Groundedness Score: 1.000
Feedback: Excellent groundedness (1.000)

Entity-Level Scores:
  Applicant Name: 1.000
  Job Title of Applicant: 1.000
  Business Registration Number of Employer: 1.000
  Height of Applicant: 1.000
  Weight of Applicant: 1.000

Numeric Entities: ['Business Registration Number of Employer', 'Height of Applicant', 'Weight of Applicant']
Textual Entities: ['Applicant Name', 'Job Title of Applicant']
Well Grounded: ['Applicant Name', 'Job Title of Applicant', 'Business Registration Number of Employer', 'Height of Applicant', 'Weight of Applicant']
Poorly Grounded: []


## 3. Semantic Expression Fidelity (SEF)

Measures how faithfully entity values are expressed in the summary:
- **Numeric-heavy entities**: Strict grounding with partial_ratio
- **Textual entities**: Fuzzy matching with token_set_ratio
- **No aggressive dampening**: Verbosity factor neutralized (1.0)

In [7]:
sef_result = sef_evaluator.evaluate(summary=summary, entities=entities)

print(f"Semantic Fidelity Score: {sef_result.score:.3f}")
print(f"Feedback: {sef_result.feedback}")
print(f"\nPer-Entity Fidelity:")
for entity, score in sef_result.metadata['entity_similarity_scores'].items():
    print(f"  {entity}: {score:.3f}")
print(f"\nAverage Semantic Fidelity: {sef_result.metadata['average_semantic_fidelity']:.3f}")
print(f"Min Similarity: {sef_result.metadata['minimum_entity_similarity']:.3f}")
print(f"Max Similarity: {sef_result.metadata['maximum_entity_similarity']:.3f}")

Semantic Fidelity Score: 1.000
Feedback: Excellent semantic fidelity (1.000)

Per-Entity Fidelity:
  Applicant Name: 1.000
  Job Title of Applicant: 1.000
  Business Registration Number of Employer: 1.000
  Height of Applicant: 1.000
  Weight of Applicant: 1.000

Average Semantic Fidelity: 1.000
Min Similarity: 1.000
Max Similarity: 1.000


## Combined Evaluation with Service

In [8]:
from evaluation.document_summarization import SummaryEvaluationService

# Initialize service
service = SummaryEvaluationService()

# Register all evaluators
service.register_evaluator(ecs_evaluator)
service.register_evaluator(gs_evaluator)
service.register_evaluator(sef_evaluator)

print(f"Registered evaluators: {service.get_registered_evaluators()}")

Registered evaluators: ['entity_coverage', 'groundedness', 'semantic_fidelity']


## Run All Evaluations

In [9]:
# Evaluate summary
result = service.evaluate(
    summary=summary,
    entities=entities,
    evaluators=["all"]  # Run all registered evaluators
)

print("="*80)
print("EVALUATION RESULTS")
print("="*80)
print(f"\nOverall Score: {result['overall_score']:.3f}")
print(f"Success: {result['success']}")

print("\nIndividual Scores:")
for evaluator_name, eval_result in result['evaluations'].items():
    status = "✓" if eval_result.get('success', True) else "✗"
    print(f"\n  {status} {evaluator_name.upper()}: {eval_result['score']:.3f}")
    print(f"    {eval_result['feedback']}")

print("\n" + "="*80)

EVALUATION RESULTS

Overall Score: 1.000
Success: True

Individual Scores:

  ✓ ENTITY_COVERAGE: 1.000
    Covered 5/5 entities (100.0%)

  ✓ GROUNDEDNESS: 1.000
    Excellent groundedness (1.000)

  ✓ SEMANTIC_FIDELITY: 1.000
    Excellent semantic fidelity (1.000)



## Calculate Final Composite Score

Weighted combination of all evaluators:
- ECS: 40%
- GS: 30%
- SEF: 230%

In [15]:
# Extract individual scores
ecs_score = result['evaluations']['entity_coverage']['score']
gs_score = result['evaluations']['groundedness']['score']
sef_score = result['evaluations']['semantic_fidelity']['score']

# Calculate weighted composite score
final_score = (
    0.40 * ecs_score +
    0.30 * gs_score +
    0.30 * sef_score
)

print("="*80)
print("FINAL COMPOSITE SCORE")
print("="*80)
print(f"\nECS (40%): {ecs_score:.3f} → {0.40 * ecs_score:.3f}")
print(f"GS  (30%): {gs_score:.3f} → {0.30 * gs_score:.3f}")
print(f"SEF (30%): {sef_score:.3f} → {0.30 * sef_score:.3f}")
print(f"\nFINAL: {final_score:.3f}")
print("="*80)

FINAL COMPOSITE SCORE

ECS (40%): 1.000 → 0.400
GS  (30%): 1.000 → 0.300
SEF (30%): 1.000 → 0.300

FINAL: 1.000


## Save Results

In [16]:
from evaluation.document_summarization.utils import save_evaluation_results

try:
    # Add final score to result
    result['final_composite_score'] = final_score
    result['weights'] = {
        "entity_coverage": 0.40,
        "groundedness": 0.30,
        "semantic_fidelity": 0.30
    }

    # Save to JSON
    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)

    json_path = output_dir / "evaluation_results.json"
    save_evaluation_results(result, str(json_path), format="json")

    print(f"✓ Results saved to {json_path}")

    # Display summary
    print("\n" + "="*80)
    print("EVALUATION SUMMARY")
    print("="*80)
    print(f"Final Composite Score: {final_score:.3f}")
    print(f"\nEvaluators Run: {len(result['evaluations'])}")
    print(f"All evaluations successful: {all(e.get('success', True) for e in result['evaluations'].values())}")
    
except Exception as e:
    print(f"❌ Error saving results: {e}")

✓ Results saved to output/evaluation_results.json

EVALUATION SUMMARY
Final Composite Score: 1.000

Evaluators Run: 3
All evaluations successful: True


## Batch Evaluation

Evaluate multiple summaries at once.

In [17]:
# Example batch data
summaries = [
    {
        "summary": summary,
        "entities": entities,
        "context": "Insurance Application"
    }
    # Add more summaries here
]

# Evaluate batch
batch_results = service.evaluate_batch(
    summaries=summaries,
    evaluators=["all"]
)

print(f"✓ Evaluated {len(batch_results)} summaries")
print()
for i, batch_result in enumerate(batch_results, 1):
    if batch_result.get('success', False):
        print(f"Summary {i}: Overall Score = {batch_result['overall_score']:.3f}")
        print(f"  Entity Coverage: {batch_result['evaluations']['entity_coverage']['score']:.3f}")
        print(f"  Groundedness: {batch_result['evaluations']['groundedness']['score']:.3f}")
        print(f"  Semantic Fidelity: {batch_result['evaluations']['semantic_fidelity']['score']:.3f}")
    else:
        print(f"Summary {i}: Evaluation failed - {batch_result.get('error_message', 'Unknown error')}")

✓ Evaluated 1 summaries

Summary 1: Overall Score = 1.000
  Entity Coverage: 1.000
  Groundedness: 1.000
  Semantic Fidelity: 1.000
